# Check age proportions in SSNAP data

In [1]:
import polars as pl
import os
import numpy as np

In [2]:
path_to_ssnap = os.path.join('..', '..', 'ssnap_data', 'clean_samuel_ssnap_extract_v2.csv')

df_ssnap = pl.read_csv(path_to_ssnap)

In [3]:
len(df_ssnap)

358993

In [4]:
df_ssnap.columns

['id',
 'stroke_team',
 'age',
 'male',
 'infarction',
 'onset_to_arrival_time',
 'onset_known',
 'precise_onset_known',
 'onset_during_sleep',
 'arrive_by_ambulance',
 'call_to_ambulance_arrival_time',
 'ambulance_on_scene_time',
 'ambulance_travel_to_hospital_time',
 'ambulance_wait_time_at_hospital',
 'month',
 'year',
 'weekday',
 'arrival_time_3_hour_period',
 'arrival_to_scan_time',
 'thrombolysis',
 'scan_to_thrombolysis_time',
 'thrombectomy',
 'arrival_to_thrombectomy_time',
 'congestive_heart_failure',
 'hypertension',
 'atrial_fibrillation',
 'diabetes',
 'prior_stroke_tia',
 'afib_antiplatelet',
 'afib_anticoagulant',
 'afib_vit_k_anticoagulant',
 'afib_doac_anticoagulant',
 'afib_heparin_anticoagulant',
 'new_afib_diagnosis',
 'prior_disability',
 'stroke_severity',
 'nihss_complete',
 'nihss_arrival_loc',
 'nihss_arrival_loc_questions',
 'nihss_arrival_loc_commands',
 'nihss_arrival_best_gaze',
 'nihss_arrival_visual',
 'nihss_arrival_facial_palsy',
 'nihss_arrival_motor_

In [5]:
welsh_teams = [
    'Prince Philip Hospital',
    'University Hospital of Wales',
    'Grange University Hospital',
    'Glan Clwyd District General Hospital',
    'West Wales General',
    'Morriston Hospital',
    'Bronglais Hospital',
    'Princess Of Wales Hospital',
    'Maelor Hospital',
    'Prince Charles Hospital',
    'Ysbyty Gwynedd',
    'Withybush General Hospital',
]

In [6]:
df_ssnap = df_ssnap.filter(~df_ssnap['stroke_team'].is_in(welsh_teams))

In [7]:
len(df_ssnap)

339814

In [8]:
len(df_ssnap.filter(df_ssnap['year'] < 2017))

53208

In [9]:
df_ssnap = df_ssnap.filter(df_ssnap['year'].is_in([2017, 2018, 2019]))

In [10]:
df_age_count = df_ssnap['age'].value_counts().sort('age')

# Combine values under 65:
mask = df_age_count['age'] < 65
n_under65 = df_age_count.filter(mask)['count'].sum()
# Remove values under 65:
df_age_count = df_age_count.filter(~mask)

# Combine values over 80:
mask = df_age_count['age'] > 80
n_over80 = df_age_count.filter(mask)['count'].sum()
# Remove values over 80:
df_age_count = df_age_count.filter(~mask)

# Add in the under 65 and over 80 values:
df_age_count = pl.concat((
    pl.DataFrame({'age': 'under65', 'count': n_under65}),
    df_age_count.cast({'age': str, 'count': int}),
    pl.DataFrame({'age': 'over80', 'count': n_over80})
))

In [11]:
df_age_count

age,count
str,i64
"""under65""",38827
"""67.5""",15324
"""72.5""",21508
"""77.5""",24150
"""over80""",67947


Calculate the proportion of these counts out of all stroke admissions:

In [12]:
n_total_admissions = df_age_count['count'].sum()

n_total_admissions

167756

In [13]:
df_age_count = df_age_count.with_columns(pl.Series('prop_of_all_admissions', df_age_count['count'] / n_total_admissions))

In [14]:
df_age_count

age,count,prop_of_all_admissions
str,i64,f64
"""under65""",38827,0.231449
"""67.5""",15324,0.091347
"""72.5""",21508,0.12821
"""77.5""",24150,0.143959
"""over80""",67947,0.405035


## Date range of SSNAP data

In [15]:
df_ssnap.sort('year', 'month')[['year', 'month']]

year,month
i64,i64
2017,1
2017,1
2017,1
2017,1
2017,1
…,…
2019,12
2019,12
2019,12


In [16]:
df_age_count = df_age_count.with_columns(pl.Series('admissions_annual', np.round(df_age_count['count'] / 3, 5)))

Data range is from January 2016 to December 2021 inclusive: six years in total.

So divide admissions by six to find the annual admissions:

In [17]:
# df_age_count = df_age_count.with_columns(pl.Series('admissions_annual', np.round(df_age_count['count'] / 6, 5)))

Boost annual admissions to match Hospital Episode Statistics data:

In [18]:
path_to_msoa_stats = os.path.join('data', 'msoa_cleaned.csv')

df_stats = pl.read_csv(path_to_msoa_stats)

In [19]:
n_admissions_hes = df_stats['admissions'].sum()
n_admissions_ssnap = df_age_count['admissions_annual'].sum()

admissions_scale = n_admissions_hes / n_admissions_ssnap

In [20]:
admissions_scale

1.447781301586404

In [22]:
df_age_count = df_age_count.with_columns(pl.Series('admissions_annual_boost', np.round(df_age_count['admissions_annual'] * admissions_scale, 5)))

In [23]:
df_age_count

age,count,prop_of_all_admissions,admissions_annual,admissions_annual_boost
str,i64,f64,f64,f64
"""under65""",38827,0.231449,12942.33333,18737.66819
"""67.5""",15324,0.091347,5108.0,7395.26689
"""72.5""",21508,0.12821,7169.33333,10379.62674
"""77.5""",24150,0.143959,8050.0,11654.63948
"""over80""",67947,0.405035,22649.0,32790.7987


Compare with population of UK by age band.

January 2019 closest to middle of data range.

In [24]:
path_to_pop = os.path.join('data', 'ukmidyearestimates20192019ladcodes.csv')

df_pop = pl.read_csv(path_to_pop)

In [25]:
df_pop

Age Groups,UNITED KINGDOM,GREAT BRITAIN,ENGLAND AND WALES,ENGLAND,WALES,SCOTLAND,NORTHERN IRELAND
str,i64,i64,i64,i64,i64,i64,i64
""" 0-4""",3857263,3736894,3465179,3299637,165542,271715,120369
""" 5-9""",4149852,4021306,3721990,3538206,183784,299316,128546
"""10-14""",3953866,3829739,3535065,3354246,180819,294674,124127
"""15-19""",3656968,3544571,3262613,3090232,172381,281958,112397
"""20-24""",4153080,4037721,3690265,3487863,202402,347456,115359
…,…,…,…,…,…,…,…
"""70-74""",3318867,3237468,2958612,2779326,179286,278856,81399
"""75-79""",2325296,2263422,2067471,1940686,126785,195951,61874
"""80-84""",1715328,1672489,1529682,1439913,89769,142807,42839


Combine under 65 and over 80:

In [26]:
df_pop['Age Groups'].to_numpy()

array([' 0-4', ' 5-9', '10-14', '15-19', '20-24', '25-29', '30-34',
       '35-39', '40-44', '45-49', '50-54', '55-59', '60-64', '65-69',
       '70-74', '75-79', '80-84', '85-89', '90 and over'], dtype=object)

In [27]:
rows_under65 = [
    ' 0-4', ' 5-9', '10-14', '15-19', '20-24', '25-29', '30-34',
    '35-39', '40-44', '45-49', '50-54', '55-59', '60-64'
]
mask_under65 = df_pop['Age Groups'].is_in(rows_under65)
df_under65 = df_pop.filter(mask_under65).sum()
df_under65 = df_under65.with_columns(pl.Series('Age Groups', ['Under 65']))

rows_over80 = ['80-84', '85-89', '90 and over']
mask_over80 = df_pop['Age Groups'].is_in(rows_over80)
df_over80 = df_pop.filter(mask_over80).sum()
df_over80 = df_over80.with_columns(pl.Series('Age Groups', ['80 and over']))

df_pop = df_pop.filter(~(mask_under65 | mask_over80))
df_pop = pl.concat((df_under65, df_pop, df_over80))

In [28]:
df_pop

Age Groups,UNITED KINGDOM,GREAT BRITAIN,ENGLAND AND WALES,ENGLAND,WALES,SCOTLAND,NORTHERN IRELAND
str,i64,i64,i64,i64,i64,i64,i64
"""Under 65""",54421846,52842903,48423748,45933245,2490503,4419155,1578943
"""65-69""",3368199,3278326,2978882,2796740,182142,299444,89873
"""70-74""",3318867,3237468,2958612,2779326,179286,278856,81399
"""75-79""",2325296,2263422,2067471,1940686,126785,195951,61874
"""80 and over""",3362599,3281021,3011127,2836964,174163,269894,81578


Only keep England. SSNAP data contains England and Wales but we removed Wales earlier.

In [29]:
col_country = 'ENGLAND'
df_pop = df_pop[['Age Groups', col_country]].rename({col_country: 'population'})

Add a column for proportion of this age band of the whole population:

In [30]:
n_total_pop = df_pop['population'].sum()

n_total_pop

56286961

In [31]:
df_pop = df_pop.with_columns(pl.Series('prop_of_all_pop', df_pop['population'] / n_total_pop))

In [32]:
df_pop

Age Groups,population,prop_of_all_pop
str,i64,f64
"""Under 65""",45933245,0.816055
"""65-69""",2796740,0.049687
"""70-74""",2779326,0.049378
"""75-79""",1940686,0.034478
"""80 and over""",2836964,0.050402


## Combine admissions and population data

In [33]:
df_age_count

age,count,prop_of_all_admissions,admissions_annual,admissions_annual_boost
str,i64,f64,f64,f64
"""under65""",38827,0.231449,12942.33333,18737.66819
"""67.5""",15324,0.091347,5108.0,7395.26689
"""72.5""",21508,0.12821,7169.33333,10379.62674
"""77.5""",24150,0.143959,8050.0,11654.63948
"""over80""",67947,0.405035,22649.0,32790.7987


In [34]:
# Rename columns to match:
rename_dict = {
    'under65': 'Under 65',
    '67.5': '65-69',
    '72.5': '70-74',
    '77.5': '75-79',
    'over80': '80 and over'
}

df_age_count = df_age_count.rename({'age': 'Age Groups'})
df_age_count = df_age_count.with_columns(df_age_count['Age Groups'].replace(rename_dict))

In [35]:
df_pop_admissions = df_pop.join(df_age_count, on='Age Groups')

In [36]:
df_pop_admissions

Age Groups,population,prop_of_all_pop,count,prop_of_all_admissions,admissions_annual,admissions_annual_boost
str,i64,f64,i64,f64,f64,f64
"""Under 65""",45933245,0.816055,38827,0.231449,12942.33333,18737.66819
"""65-69""",2796740,0.049687,15324,0.091347,5108.0,7395.26689
"""70-74""",2779326,0.049378,21508,0.12821,7169.33333,10379.62674
"""75-79""",1940686,0.034478,24150,0.143959,8050.0,11654.63948
"""80 and over""",2836964,0.050402,67947,0.405035,22649.0,32790.7987


In [37]:
df_pop_admissions.sum()

Age Groups,population,prop_of_all_pop,count,prop_of_all_admissions,admissions_annual,admissions_annual_boost
str,i64,f64,i64,f64,f64,f64
null,56286961,1.0,167756,1.0,55918.66666,80958.0


Given that you're in an age band, what is the probability of having a stroke?

In [38]:
df_pop_admissions = df_pop_admissions.with_columns(pl.Series('prob_stroke_given_age', df_pop_admissions['admissions_annual_boost'] / df_pop_admissions['population']))

In [39]:
df_pop_admissions

Age Groups,population,prop_of_all_pop,count,prop_of_all_admissions,admissions_annual,admissions_annual_boost,prob_stroke_given_age
str,i64,f64,i64,f64,f64,f64,f64
"""Under 65""",45933245,0.816055,38827,0.231449,12942.33333,18737.66819,0.000408
"""65-69""",2796740,0.049687,15324,0.091347,5108.0,7395.26689,0.002644
"""70-74""",2779326,0.049378,21508,0.12821,7169.33333,10379.62674,0.003735
"""75-79""",1940686,0.034478,24150,0.143959,8050.0,11654.63948,0.006005
"""80 and over""",2836964,0.050402,67947,0.405035,22649.0,32790.7987,0.011558


In [40]:
df_pop_admissions[['Age Groups', 'prop_of_all_pop', 'admissions_annual_boost', 'prob_stroke_given_age']]

Age Groups,prop_of_all_pop,admissions_annual_boost,prob_stroke_given_age
str,f64,f64,f64
"""Under 65""",0.816055,18737.66819,0.000408
"""65-69""",0.049687,7395.26689,0.002644
"""70-74""",0.049378,10379.62674,0.003735
"""75-79""",0.034478,11654.63948,0.006005
"""80 and over""",0.050402,32790.7987,0.011558


In [41]:
df_pop_admissions[['population', 'admissions_annual_boost']].sum()

population,admissions_annual_boost
i64,f64
56286961,80958.0


In [42]:
df_pop_admissions['prob_stroke_given_age']# * df_pop_admissions['population']

prob_stroke_given_age
f64
0.000408
0.002644
0.003735
0.006005
0.011558


In [43]:
[4.5596e7,	2.7612e6,	2.7912e6,	1.9936e6,	2.8329e6]

[45596000.0, 2761200.0, 2791200.0, 1993600.0, 2832900.0]

In [44]:
(55918.66666 / 9) * 10

62131.85184444445